<i18n value="d2c3cdc2-5fcf-4edc-8101-2964a9355000"/>

# 데이터 추출 및 로드 실습

이 실습에서는 JSON 파일에서 원시 데이터를 추출하여 Delta 테이블에 로드합니다.

## 학습 목표
이 실습을 마치면 다음을 수행할 수 있습니다.
- JSON 파일에서 데이터를 추출하는 외부 테이블 생성
- 제공된 스키마를 사용하여 빈 Delta 테이블 생성
- 기존 테이블의 레코드를 Delta 테이블에 삽입
- CTAS 문을 사용하여 파일에서 Delta 테이블 생성

<i18n value="e261fd97-ffd7-44b2-b1ca-61b843ee8961"/>

## 실행 설정

이 레슨의 변수와 데이터세트를 구성하려면 다음 셀을 실행하세요.

In [0]:
%run ../Includes/Classroom-Setup-04.5L

Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(2 seconds)

Predefined tables in "3dt005_nuk5_da_dewd":
| -none-

Predefined paths variables:
| DA.paths.working_dir: dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks
| DA.paths.user_db:     dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks/database.db
| DA.paths.datasets:    dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02
| DA.paths.checkpoints: dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks/_checkpoints

Setup completed (6 seconds)


<i18n value="d7759322-f9b9-4abe-9b30-25f5a7e30d9c"/>

## 데이터 개요

JSON 파일로 작성된 원시 Kafka 데이터 샘플을 사용합니다.

각 파일에는 5초 간격 동안 사용된 모든 레코드가 포함되어 있으며, 전체 Kafka 스키마와 함께 다중 레코드 JSON 파일로 저장됩니다.

테이블 스키마는 다음과 같습니다.

| field | type | description |
| ------ | ---- | ----------- |
| key | BINARY | **`user_id`** 필드는 키로 사용됩니다. 세션/쿠키 정보에 해당하는 고유한 영숫자 필드입니다. |
| offset | LONG | 각 파티션에 대해 단조롭게 증가하는 고유한 값입니다. |
| partition | INTEGER | 현재 Kafka 구현에서는 2개의 파티션(0과 1)만 사용합니다. |
| timestamp | LONG | 이 타임스탬프는 에포크 이후 밀리초 단위로 기록되며, 프로듀서가 파티션에 레코드를 추가하는 시간을 나타냅니다. |
| topic | STRING | Kafka 서비스는 여러 토픽을 호스팅하지만, **`clickstream`** 토픽의 레코드만 여기에 포함됩니다. |
| value | BINARY | 전체 데이터 페이로드(나중에 설명)이며, JSON 형식으로 전송됩니다.

<i18n value="f2cd70fe-65a1-4dce-b264-c0c7d225640a"/>

## JSON 파일에서 원시 이벤트 추출
이 데이터를 Delta에 제대로 로드하려면 먼저 올바른 스키마를 사용하여 JSON 데이터를 추출해야 합니다.

아래 제공된 파일 경로에 있는 JSON 파일에 대한 외부 테이블을 생성합니다. 이 테이블의 이름을 **`events_json`**으로 지정하고 위의 스키마를 선언합니다.

In [0]:
CREATE TABLE IF NOT EXISTS events_json(
  key BINARY,
  offset BIGINT,
  partition INT,
  timestamp BIGINT,
  topic STRING,
  value BINARY
)
USING JSON
OPTIONS (path='${da.paths.datasets}/ecommerce/raw/events-kafka/')

<i18n value="07ce3850-fdc7-4dea-9335-2a093c2e200c"/>

**참고**: 랩 전체에서 Python을 사용하여 가끔씩 검사를 실행합니다. 지침을 따르지 않은 경우 다음 셀에서 변경해야 할 사항에 대한 메시지와 함께 오류가 반환됩니다. 셀 실행 결과 출력이 없으면 이 단계가 완료되었음을 의미합니다.

In [0]:
%python
assert spark.table("events_json"), "Table named `events_json` does not exist"
assert spark.table("events_json").columns == ['key', 'offset', 'partition', 'timestamp', 'topic', 'value'], "Please name the columns in the order provided above"
assert spark.table("events_json").dtypes == [('key', 'binary'), ('offset', 'bigint'), ('partition', 'int'), ('timestamp', 'bigint'), ('topic', 'string'), ('value', 'binary')], "Please make sure the column types are identical to those provided above"

total = spark.table("events_json").count()
assert total == 2252, f"Expected 2252 records, found {total}"

<i18n value="ae3b8554-d0e7-4fd7-b25a-27bfbc5f7c13"/>

## 델타 테이블에 원시 이벤트 삽입
동일한 스키마를 사용하여 **`events_raw`**라는 이름의 빈 관리형 델타 테이블을 생성합니다.

In [0]:
CREATE OR REPLACE TABLE events_raw(
  key BINARY,
  offset BIGINT,
  partition INT,
  timestamp BIGINT,
  topic STRING,
  value BINARY
)
USING DELTA;

--DESCRIBE EXTENDED events_raw;
-- ('topic', 'string'), ('value', 'binary') AS
--SELECT * FROM JSON.`${da.paths.datasets}/ecommerce/raw/events-kafka/`;

DESCRIBE EXTENDED events_raw;

col_name,data_type,comment
key,binary,null
offset,bigint,null
partition,int,null
timestamp,bigint,null
topic,string,null
value,binary,null
,,
# Detailed Table Information,,
Catalog,hive_metastore,
Database,3dt005_nuk5_da_dewd,


<i18n value="3d56975b-47ba-4678-ae7b-7c5e4ac20a97"/>

아래 셀을 실행하여 테이블이 올바르게 생성되었는지 확인하세요.

In [0]:
%python
assert spark.table("events_raw"), "Table named `events_raw` does not exist"
assert spark.table("events_raw").columns == ['key', 'offset', 'partition', 'timestamp', 'topic', 'value'], "Please name the columns in the order provided above"
assert spark.table("events_raw").dtypes == [('key', 'binary'), ('offset', 'bigint'), ('partition', 'int'), ('timestamp', 'bigint'), ('topic', 'string'), ('value', 'binary')], "Please make sure the column types are identical to those provided above"
assert spark.table("events_raw").count() == 0, "The table should have 0 records"

<i18n value="61815a62-6d4f-47fb-98a9-73c39842ac56"/>

추출된 데이터와 Delta 테이블이 준비되면 **`events_json`** 테이블의 JSON 레코드를 새 **`events_raw`** Delta 테이블에 삽입합니다.

In [0]:
-- TODO
INSERT OVERWRITE events_raw
-- INSERT INTO events_raw
SELECT * FROM events_json;

num_affected_rows,num_inserted_rows
2252,2252


<i18n value="4f545052-31c6-442b-a5e8-4c5892ec912f"/>

테이블 내용을 직접 검토하여 데이터가 예상대로 작성되었는지 확인하세요.

In [0]:
-- TODO
SELECT * FROM events_raw;

key,offset,partition,timestamp,topic,value
VUEwMDAwMDAxMDczOTgwNTQ=,219255030,0,1593880885085,clickstream,eyJkZXZpY2UiOiJBbmRyb2lkIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6Im1haW4iLCJldmVudF90aW1lc3RhbXAiOjE1OTM4ODA4ODUwMzYxMjksImdlbyI6eyJjaXR5IjoiTmV3IFlvcmsiLCJzdGF0ZSI6Ik5ZIn0= (truncated)
VUEwMDAwMDAxMDczOTI0NTg=,219255043,0,1593880892303,clickstream,eyJkZXZpY2UiOiJpT1MiLCJlY29tbWVyY2UiOnt9LCJldmVudF9uYW1lIjoiYWRkX2l0ZW0iLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOjE1OTM4ODAzMDA2OTY3NTEsImV2ZW50X3RpbWVzdGFtcCI6MTU5Mzg4MDg5MjI= (truncated)
VUEwMDAwMDAxMDczOTU5Njg=,219255108,0,1593880889174,clickstream,eyJkZXZpY2UiOiJtYWNPUyIsImVjb21tZXJjZSI6e30sImV2ZW50X25hbWUiOiJwcmVtaXVtIiwiZXZlbnRfcHJldmlvdXNfdGltZXN0YW1wIjoxNTkzODgwODYxMDMwMjQxLCJldmVudF90aW1lc3RhbXAiOjE1OTM4ODA4ODk= (truncated)
VUEwMDAwMDAxMDczOTgwMzA=,219255118,0,1593880889725,clickstream,eyJkZXZpY2UiOiJpT1MiLCJlY29tbWVyY2UiOnt9LCJldmVudF9uYW1lIjoib3JpZ2luYWwiLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOjE1OTM4ODA4ODI0Mjk5ODAsImV2ZW50X3RpbWVzdGFtcCI6MTU5Mzg4MDg4OTY= (truncated)
VUEwMDAwMDAxMDczODIyMzM=,219438025,1,1593880886106,clickstream,eyJkZXZpY2UiOiJBbmRyb2lkIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6ImNjX2luZm8iLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOjE1OTM4ODAzNjQzMjEwODgsImV2ZW50X3RpbWVzdGFtcCI6MTU5Mzg4MDg= (truncated)
VUEwMDAwMDAxMDczODIyMzM=,219438069,1,1593880886106,clickstream,eyJkZXZpY2UiOiJBbmRyb2lkIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6ImNjX2luZm8iLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOjE1OTM4ODAzNjQzMjEwODgsImV2ZW50X3RpbWVzdGFtcCI6MTU5Mzg4MDg= (truncated)
VUEwMDAwMDAxMDczOTgwMzc=,219438089,1,1593880887640,clickstream,eyJkZXZpY2UiOiJBbmRyb2lkIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6ImRlbGl2ZXJ5IiwiZXZlbnRfcHJldmlvdXNfdGltZXN0YW1wIjoxNTkzODgwODgyOTY0MjYyLCJldmVudF90aW1lc3RhbXAiOjE1OTM4ODA= (truncated)
VUEwMDAwMDAxMDczOTgxNTk=,219438114,1,1593880894803,clickstream,eyJkZXZpY2UiOiJtYWNPUyIsImVjb21tZXJjZSI6e30sImV2ZW50X25hbWUiOiJtYWluIiwiZXZlbnRfdGltZXN0YW1wIjoxNTkzODgwODk0Nzg5NTc5LCJnZW8iOnsiY2l0eSI6Ikxha2V3b29kIiwic3RhdGUiOiJDTyJ9LCI= (truncated)
VUEwMDAwMDAxMDczNzY0Njc=,219438126,1,1593880888445,clickstream,eyJkZXZpY2UiOiJXaW5kb3dzIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6ImNhcnQiLCJldmVudF9wcmV2aW91c190aW1lc3RhbXAiOjE1OTM4Nzk2MTk4NTI2NzgsImV2ZW50X3RpbWVzdGFtcCI6MTU5Mzg4MDg4ODM= (truncated)
VUEwMDAwMDAxMDczOTgwMzc=,219438135,1,1593880887640,clickstream,eyJkZXZpY2UiOiJBbmRyb2lkIiwiZWNvbW1lcmNlIjp7fSwiZXZlbnRfbmFtZSI6ImRlbGl2ZXJ5IiwiZXZlbnRfcHJldmlvdXNfdGltZXN0YW1wIjoxNTkzODgwODgyOTY0MjYyLCJldmVudF90aW1lc3RhbXAiOjE1OTM4ODA= (truncated)


<i18n value="0d66f26b-3df6-4819-9d84-22da9f55aeaa"/>

아래 셀을 실행하여 데이터가 올바르게 로드되었는지 확인하세요.

In [0]:
%python
import pyspark.sql.functions as F
assert spark.table("events_raw").count() == 2252, "The table should have 2252 records"

first_5 = [row['timestamp'] for row in spark.table("events_raw").select("timestamp").orderBy(F.col("timestamp").asc()).limit(5).collect()]
assert first_5 == [1593879303631, 1593879304224, 1593879305465, 1593879305482, 1593879305746], "Make sure you have not modified the data provided"

last_5 = [row['timestamp'] for row in spark.table("events_raw").select("timestamp").orderBy(F.col("timestamp").desc()).limit(5).collect()]
assert last_5 == [1593881096290, 1593881095799, 1593881093452, 1593881093394, 1593881092076], "Make sure you have not modified the data provided"

<i18n value="e9565088-1762-4f89-a06f-49576a53526a"/>

## 쿼리에서 델타 테이블 생성
새로운 이벤트 데이터 외에도, 나중에 사용할 제품 세부 정보를 제공하는 작은 조회 테이블도 로드해 보겠습니다.
CTAS ​​문을 사용하여 아래에 제공된 Parquet 디렉터리에서 데이터를 추출하는 **`item_lookup`**이라는 관리형 델타 테이블을 생성합니다.

In [0]:
-- TODO
CREATE OR REPLACE TABLE item_lookup
USING DELTA
AS SELECT * FROM parquet.`${da.paths.datasets}/ecommerce/raw/item-lookup/`;

num_affected_rows,num_inserted_rows


<i18n value="9f1ad20f-1238-4a12-ad2a-f10169ed6475"/>

아래 셀을 실행하여 조회 테이블이 올바르게 로드되었는지 확인하세요.

In [0]:
%python
assert spark.table("item_lookup").count() == 12, "The table should have 12 records"
assert set(row['item_id'] for row in spark.table("item_lookup").select("item_id").orderBy('item_id').limit(5).collect()) == {'M_PREM_F', 'M_PREM_K', 'M_PREM_Q', 'M_PREM_T', 'M_STAN_F'}, "Make sure you have not modified the data provided"

<i18n value="c24885ea-010e-4b76-9e9d-cc749f10993a"/>

이 레슨과 관련된 표와 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
%python
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_dewd"...(2 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(2 seconds)
| validation completed...(2 seconds total)
